# ⚡ PyArrow-Backed Pandas Support

Articuno fully supports PyArrow-backed Pandas columns for high performance.

In [1]:
from articuno import infer_pydantic_model, df_to_pydantic
import pandas as pd
import pyarrow as pa
from datetime import datetime, date, timedelta

## Basic PyArrow Types

In [2]:
# Create DataFrame with PyArrow-backed columns
df = pd.DataFrame(
    {
        "id": pd.Series([1, 2, 3], dtype="int64[pyarrow]"),
        "name": pd.Series(["Alice", "Bob", "Charlie"], dtype="string[pyarrow]"),
        "score": pd.Series([95.5, 88.0, 92.3], dtype="float64[pyarrow]"),
        "active": pd.Series([True, False, True], dtype="bool[pyarrow]"),
    }
)

print(df.dtypes)

id         int64[pyarrow]
name      string[pyarrow]
score     double[pyarrow]
active      bool[pyarrow]
dtype: object


In [3]:
# Infer model
ArrowModel = infer_pydantic_model(df, model_name="ArrowUser")
instances = list(df_to_pydantic(df, model=ArrowModel))

print("Instances:")
for inst in instances:
    print(f"  {inst}")

Instances:
  id=1 name='Alice' score=95.5 active=True
  id=2 name='Bob' score=88.0 active=False
  id=3 name='Charlie' score=92.3 active=True


## PyArrow Temporal Types

In [4]:
# PyArrow timestamp, date, and duration types
df_temporal = pd.DataFrame(
    {
        "id": [1, 2, 3],
        "created": pd.Series(
            [
                datetime(2024, 1, 1, 10, 30),
                datetime(2024, 1, 2, 14, 45),
                datetime(2024, 1, 3, 9, 15),
            ],
            dtype=pd.ArrowDtype(pa.timestamp("ms")),
        ),
        "birth_date": pd.Series(
            [date(1990, 5, 15), date(1985, 8, 22), date(1992, 12, 10)],
            dtype=pd.ArrowDtype(pa.date32()),
        ),
        "session_duration": pd.Series(
            [
                timedelta(hours=2, minutes=30),
                timedelta(hours=1, minutes=45),
                timedelta(hours=3, minutes=15),
            ],
            dtype=pd.ArrowDtype(pa.duration("ms")),
        ),
    }
)

print(df_temporal.dtypes)

id                                   int64
created             timestamp[ms][pyarrow]
birth_date            date32[day][pyarrow]
session_duration     duration[ms][pyarrow]
dtype: object


In [5]:
TemporalModel = infer_pydantic_model(df_temporal, model_name="TemporalRecord")

print("Model fields:")
for name, field in TemporalModel.model_fields.items():
    print(f"  {name}: {field.annotation}")

instances = list(df_to_pydantic(df_temporal, model=TemporalModel))
print(f"\nCreated {len(instances)} instances with PyArrow temporal types")

Model fields:
  id: <class 'int'>
  created: <class 'datetime.datetime'>
  birth_date: <class 'datetime.datetime'>
  session_duration: typing.Any

Created 3 instances with PyArrow temporal types


## Benefits of PyArrow

- ✅ Higher performance
- ✅ Better null handling
- ✅ Memory efficiency
- ✅ Full articuno support